# OSPool workshop.copy-paste guide

**Instructor: Samah Alshrief**

This notebook is your paste buffer. Every code block below is either:
- **A shell command**  -  paste into your OSPool terminal (`ap40.uw.osg-htc.org`)
- **File contents**  -  you'll create the file with `nano` and paste the block inside

Nothing runs from inside this notebook. Read → run in Terminal → come back for the next step.

## 0. SSH into the OSPool access point

In your laptop's terminal:

In [ ]:
# In your LAPTOP terminal:
ssh <your-osg-username>@ap40.uw.osg-htc.org

## 1. Clone the workshop repo

Once you're on `ap40`:

In [ ]:
git clone https://github.com/Samah989/osg-workshop-ipeds.git ~/workshop
cd ~/workshop
ls
# Expected: README.md, run_regression.py, requirements-osg.txt, src/, data/, notebooks/, ANSWER_KEY/

## 2. Peek at what we're working with

In [ ]:
# The data: 5 parquet files, one per IPEDS dataset, one year (2015).
ls -lh data/

# The analysis: reads parquets, fits a fixed-effects panel regression, writes coefficients CSV.
head -30 run_regression.py

# The helper module.
head -30 src/regression.py

## 3. Write the shell wrapper  -  `run_regression.sh`

The wrapper is what HTCondor actually runs inside the container. It installs Python deps, then calls your analysis.

Open a new file with nano:

In [ ]:
nano run_regression.sh

Paste this in. Save with `Ctrl-O`, `Enter`, `Ctrl-X`.

```bash
#!/bin/bash
# OSG wrapper: install deps, run the regression.
set -eo pipefail
export PS4='+ [regress $(date +%H:%M:%S)] '
set -x

# 1. Create results/ FIRST so transfer_output_files always has a dir.
mkdir -p results deps
: > results/.placeholder

echo "[regress] host=$(hostname)  start=$(date -Iseconds)"
ls -la data 2>/dev/null | head -10

# 2. Install Python deps into ./deps (osgvo-ubuntu-20.04 lacks python3-venv).
python3 -m ensurepip --user 2>/dev/null || true
python3 -m pip install --upgrade --target=deps pip 2>/dev/null || python3 -m pip install --user --upgrade pip
python3 -m pip install --target=deps -r requirements-osg.txt

# 3. Put ./deps on the import path and run.
export PYTHONPATH="$PWD/deps:${PYTHONPATH:-}"
python3 run_regression.py --out results/

echo "[regress] done=$(date -Iseconds)  outputs:"
ls -lh results
```

Then make it executable:

In [ ]:
chmod +x run_regression.sh

## 4. Write the HTCondor submit file  -  `run_regression.submit`

The submit file tells HTCondor: what container, what to run, what to ship up, what to bring back, how much CPU/memory.

In [ ]:
nano run_regression.submit

Paste this in:

```
# What container to use  -  Ubuntu 20.04 from the OSG CVMFS repo.
container_image = /cvmfs/singularity.opensciencegrid.org/opensciencegrid/osgvo-ubuntu-20.04:latest

# What to run.
executable = run_regression.sh

# What to ship UP with the job, and what to bring BACK when it exits.
transfer_input_files    = run_regression.py, requirements-osg.txt, src, data
transfer_output_files   = results
should_transfer_files   = YES
when_to_transfer_output = ON_EXIT

# Resource request  -  modest so OSPool matches faster.
request_cpus   = 2
request_memory = 4 GB
request_disk   = 4 GB

# OSPool duration category. Medium = up to 10 hours wallclock.
+JobDurationCategory = "Medium"

# Retry if evicted.
max_retries = 2

# Log everything for debugging.
log             = logs/regress.$(Cluster).log
output          = logs/regress.$(Cluster).out
error           = logs/regress.$(Cluster).err
stream_output   = True
stream_error    = True

# Queue one instance.
queue 1
```

Save (`Ctrl-O`, `Enter`, `Ctrl-X`) and make the logs directory:

In [ ]:
mkdir -p logs

## 5. Submit the job

In [ ]:
condor_submit run_regression.submit
# Expected: "1 job(s) submitted to cluster NNNNNN."
# Remember the cluster number.

## 6. Watch the queue

In [ ]:
condor_q -nobatch
# Column meanings:
#   ST = job state:  I=idle (waiting for slot)   R=running   H=held (something broke)   C=completed
#   RUN_TIME = days+HH:MM:SS on the execute host

# When the job leaves the queue (no longer listed) it either finished or failed. Check exit code:
condor_history <CLUSTER> -limit 1 -af ExitCode HoldReason JobStatus

## 7. Read the logs

Three files land in `logs/`:
- `regress.<CLUSTER>.log`  -  HTCondor event log (submitted / started / evicted / completed)
- `regress.<CLUSTER>.out`  -  stdout of your job (the `print()` output)
- `regress.<CLUSTER>.err`  -  stderr (the `set -x` trace + any Python tracebacks)

In [ ]:
tail -n 30 logs/regress.<CLUSTER>.out
tail -n 30 logs/regress.<CLUSTER>.err

## 8. Read the result

In [ ]:
cat results/rq2_main.csv
# You should see coefficient rows for log_instr_per_fte, log_admin_per_fte, log_fte, sector_2, sector_3.

## 9. Pull the result back to your laptop

Log OUT of ap40 (`exit`), then from your **laptop terminal**:

In [ ]:
rsync -avz <your-osg-username>@ap40.uw.osg-htc.org:'~/workshop/results/' ./workshop-results/
ls workshop-results/

## Troubleshooting

| Symptom | Fix |
|---|---|
| Job stays `I` for many minutes | Ask smaller resources; verify `+JobDurationCategory` is `Medium` or `Long` (not `Short`) |
| Job goes `H` (held) with "transfer output files failure" | Your wrapper crashed before creating `results/`. Fix: `mkdir -p results` at the TOP of the wrapper |
| `.err` shows `No module named 'linearmodels'` | Missing from `requirements-osg.txt`, or pip install failed silently |
| `KeyError: 'X'` in the Python traceback | Your script expects a column that isn't in the parquets  -  check names |
| Container path errors | Confirm the CVMFS path: `osgvo-ubuntu-20.04:latest` (NOT `22.04`  -  that image doesn't exist) |

## Cleanup after the workshop

```
condor_rm <CLUSTER>          # remove any leftover jobs
rm -rf ~/workshop/logs/*     # keep home tidy
```

## Where to go next

- The `ANSWER_KEY/` folder in the repo has the final `.sh` and `.submit` files
- Try adding a second RQ as a queued sub-job: change `queue 1` to `queue rq_number from ( 1  2  3 )` and read `$(rq_number)` in the wrapper
- OSPool docs: https://portal.osg-htc.org/documentation/
- HTCondor manual: https://htcondor.readthedocs.io